# Student Placement Prediction using Machine Learning

## IBM SkillsBuild Data Analytics with AI Internship 2026

**Project type:** Supervised Machine Learning — Binary Classification  
**Goal:** Analyze student academic/profile data and predict whether a student is likely to be placed.

> **Dataset note:** This project uses a **synthetically generated dataset** inside the notebook so that it can run without downloading a private or restricted dataset. The synthetic data is created only for educational demonstration and does not represent real students or real placement decisions.


## 1. Project Aim

To build an end-to-end AI/ML workflow that:
- creates and explores a student dataset,
- performs basic data analytics and visualization,
- preprocesses numerical and categorical features,
- trains classification models,
- evaluates model performance,
- compares models, and
- demonstrates a prediction for a new student.

### Key question
Can academic performance, attendance, skills, projects, internships, and study habits be used to predict a student's placement outcome in a synthetic educational dataset?


In [ ]:
# Install (if needed):
# pip install -r requirements.txt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded successfully.")


## 2. Generate the Synthetic Dataset

In [ ]:
# Create a reproducible synthetic dataset
n = 1000
rng = np.random.default_rng(RANDOM_STATE)

cgpa = np.clip(rng.normal(7.4, 1.0, n), 5.0, 10.0)
attendance = np.clip(rng.normal(78, 10, n), 45, 100)
study_hours = np.clip(rng.normal(3.5, 1.5, n), 0.5, 10)
projects = np.clip(rng.poisson(2.2, n), 0, 6)
internships = np.clip(rng.poisson(0.7, n), 0, 3)
coding_score = np.clip(rng.normal(65, 15, n), 20, 100)

department = rng.choice(
    ["CSE", "IT", "ECE", "EEE", "ME"],
    size=n,
    p=[0.30, 0.25, 0.18, 0.15, 0.12]
)

communication = rng.choice(
    ["Low", "Medium", "High"],
    size=n,
    p=[0.20, 0.55, 0.25]
)

# Latent placement score used only to generate the synthetic target
comm_score = pd.Series(communication).map({"Low": 0, "Medium": 1, "High": 2}).to_numpy()
department_bonus = pd.Series(department).map(
    {"CSE": 0.25, "IT": 0.20, "ECE": 0.10, "EEE": 0.0, "ME": -0.05}
).to_numpy()

latent_score = (
    1.15 * (cgpa - 7)
    + 0.025 * (attendance - 75)
    + 0.18 * study_hours
    + 0.35 * projects
    + 0.55 * internships
    + 0.018 * (coding_score - 60)
    + 0.35 * comm_score
    + department_bonus
    + rng.normal(0, 0.9, n)
)

placement_probability = 1 / (1 + np.exp(-latent_score))
placed = rng.binomial(1, placement_probability)

df = pd.DataFrame({
    "CGPA": np.round(cgpa, 2),
    "Attendance": np.round(attendance, 1),
    "Study_Hours_Per_Day": np.round(study_hours, 1),
    "Projects": projects,
    "Internships": internships,
    "Coding_Score": np.round(coding_score, 1),
    "Department": department,
    "Communication": communication,
    "Placed": placed
})

# Add a small amount of missing data to demonstrate preprocessing
for col in ["CGPA", "Attendance", "Coding_Score"]:
    idx = rng.choice(df.index, size=10, replace=False)
    df.loc[idx, col] = np.nan

print("Dataset shape:", df.shape)
df.head()


## 3. Data Understanding

In [ ]:
print("Data types and non-null counts:")
print(df.info())

print("\nSummary statistics:")
display(df.describe(include="all").T)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["Placed"].value_counts())


## 4. Exploratory Data Analysis

In [ ]:
# Target distribution
df["Placed"].value_counts().sort_index().plot(kind="bar")
plt.title("Placement Outcome Distribution")
plt.xlabel("Placed (0 = No, 1 = Yes)")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()

# CGPA vs placement
df.boxplot(column="CGPA", by="Placed")
plt.title("CGPA by Placement Outcome")
plt.suptitle("")
plt.xlabel("Placed")
plt.ylabel("CGPA")
plt.tight_layout()
plt.show()

# Average numerical features by placement
numeric_cols = ["CGPA", "Attendance", "Study_Hours_Per_Day", "Projects", "Internships", "Coding_Score"]
group_means = df.groupby("Placed")[numeric_cols].mean().T
group_means.plot(kind="bar", figsize=(10, 5))
plt.title("Average Student Features by Placement Outcome")
plt.ylabel("Average value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Department distribution
pd.crosstab(df["Department"], df["Placed"], normalize="index").plot(kind="bar", figsize=(8, 5))
plt.title("Placement Rate by Department")
plt.ylabel("Proportion")
plt.xlabel("Department")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Prepare Features and Target

In [ ]:
X = df.drop(columns="Placed")
y = df["Placed"]

numeric_features = [
    "CGPA", "Attendance", "Study_Hours_Per_Day",
    "Projects", "Internships", "Coding_Score"
]
categorical_features = ["Department", "Communication"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 6. Train Two Machine Learning Models

In [ ]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250,
        max_depth=10,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

logistic_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print("Both models trained successfully.")


## 7. Evaluate the Models

In [ ]:
def evaluate_model(name, model):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", logistic_model),
    evaluate_model("Random Forest", rf_model)
])

display(results.round(3))


## 8. Confusion Matrix and Classification Report

In [ ]:
# Use Random Forest for an illustrative detailed evaluation.
rf_pred = rf_model.predict(X_test)

print("Random Forest classification report:")
print(classification_report(y_test, rf_pred, target_names=["Not Placed", "Placed"]))

ConfusionMatrixDisplay.from_predictions(
    y_test, rf_pred, display_labels=["Not Placed", "Placed"]
)
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()


## 9. ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(
    y_test, logistic_model.predict_proba(X_test)[:, 1], name="Logistic Regression"
)
RocCurveDisplay.from_predictions(
    y_test, rf_model.predict_proba(X_test)[:, 1], name="Random Forest"
)
plt.title("ROC Curves")
plt.tight_layout()
plt.show()


## 10. Example Prediction for a New Student

In [ ]:
new_student = pd.DataFrame([{
    "CGPA": 8.2,
    "Attendance": 86,
    "Study_Hours_Per_Day": 4.5,
    "Projects": 3,
    "Internships": 1,
    "Coding_Score": 78,
    "Department": "IT",
    "Communication": "High"
}])

prediction = rf_model.predict(new_student)[0]
probability = rf_model.predict_proba(new_student)[0, 1]

print("Predicted class:", "Placed" if prediction == 1 else "Not Placed")
print(f"Predicted placement probability: {probability:.2%}")


## 11. Conclusion

This project demonstrates a complete data analytics and machine-learning pipeline:

1. Synthetic educational data generation
2. Data inspection and missing-value analysis
3. Exploratory data analysis
4. Feature preprocessing
5. Model training
6. Model evaluation using Accuracy, Precision, Recall, F1-score and ROC-AUC
7. Confusion matrix and ROC curve visualization
8. Prediction for a new record

### Important limitation
The dataset is synthetic and the target is generated from an artificial relationship. Therefore, model performance **must not be interpreted as evidence about real-world student placement outcomes**. For a real deployment, a properly collected, representative and consented dataset would be required, along with fairness, privacy and validation checks.


## 12. Viva / Presentation Points

**Problem:** Students and institutions may want to understand which measurable academic/profile factors are associated with a placement outcome.

**AI technique:** Supervised binary classification.

**Models:** Logistic Regression and Random Forest.

**Why preprocessing?** Missing values need imputation, numerical variables benefit from scaling for Logistic Regression, and categorical variables must be converted into machine-readable features.

**Evaluation:** Accuracy, Precision, Recall, F1-score, ROC-AUC and confusion matrix.

**Future scope:** Replace synthetic data with an approved real dataset; add cross-validation, hyperparameter tuning, explainability (SHAP), fairness checks, and a simple Streamlit dashboard.
